In [0]:
from sklearn.cluster import KMeans
import mlflow.sklearn

df = spark.table("instagram.goldlayer.vw_social_lifestyle_impact").toPandas()

# Use social/lifestyle detailing for clustering
features = ['followers_count', 'travel_frequency_per_year', 'hobbies_count', 'social_events_per_month']
X = df[features]

with mlflow.start_run(run_name="User_Segmentation"):
    kmeans = KMeans(n_clusters=4, random_state=42).fit(X)
    mlflow.sklearn.log_model(kmeans, "model")
    
    # Save cluster IDs
    predictions = df[['user_id']].copy()
    predictions['user_segment_id'] = kmeans.labels_
    spark.createDataFrame(predictions).write.mode("overwrite").saveAsTable("instagram.model_output.ml_segments")

print("✅ User Segments Generated, bro!")